In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

c:\Users\Jing Yao\Desktop\School Files\BT4014\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Import data 
news_articles = pd.read_csv(
    r"C:\Users\Jing Yao\Desktop\School Files\BT4014\Project\MINDsmall_train\news.tsv",
    sep="\t",
    header=None
)
news_articles.columns = ['News_ID', 
                         'Category', 
                         'Subcategory',
                         'Title',
                         'Abstract',
                         'URL',
                         'Title Entities',
                         'Abstract Entities']

users_interaction = pd.read_csv(
    r"C:\Users\Jing Yao\Desktop\School Files\BT4014\Project\MINDsmall_train\behaviors.tsv",
    sep="\t",
    header=None
)
users_interaction.columns = ['No.',
                            'User_ID',
                            'Time_Stamp',
                            'History',
                            'Impression']

In [ ]:
# Feature Engineering: Transforming Article Features into Embeddings
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

news_articles["text_for_embedding"] = (
    "Category: " + news_articles["Category"].fillna("") + ". " +
    "Subcategory: " + news_articles["Subcategory"].fillna("") + ". " +
    "Title: " + news_articles["Title"].fillna("") + ". " +
    "Abstract: " + news_articles["Abstract"].fillna("")
)

# article_embeddings = model.encode(
#     news_articles["text_for_embedding"].tolist(),
#     show_progress_bar=True
# )

article_embeddings = np.load(r"C:\Users\Jing Yao\Desktop\School Files\BT4014\Project\MINDsmall_train\article_embeddings.npy")
article_embedding_dict = {
    news_id: emb
    for news_id, emb in zip(news_articles["News_ID"], article_embeddings)
}

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3242.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Helper Functions
def get_user_embedding(history, article_embedding_dict, embedding_dim=384):
    if pd.isna(history) or history == "":
        return np.zeros(embedding_dim)
    
    article_ids = history.split()
    embs = [
        article_embedding_dict[aid]
        for aid in article_ids
        if aid in article_embedding_dict
    ]
    
    if len(embs) == 0:
        return np.zeros(embedding_dim)
    
    return np.mean(embs, axis=0)

def make_context(user_embedding, article_embedding):
    return np.concatenate([user_embedding, article_embedding])

In [ ]:
# EpsilonGreedy
class DecayingEpsilonGreedyLinear:
    def __init__(self, d, initial_epsilon=1.0, decay_rate=0.001):
        self.name = "DecayingEpsilonGreedyLinear"
        self.d = d
        self.theta = np.zeros(d)

        self.initial_epsilon = initial_epsilon
        self.decay_rate = decay_rate

        self.t = 1  # timestep counter

    def get_epsilon(self):
        epsilon = self.initial_epsilon / (1 + self.decay_rate * self.t)
        return epsilon

    def predict(self, x):
        return np.dot(self.theta, x)

    def select_arm(self, user_features, candidate_article_ids, article_embedding_dict):
        epsilon = self.get_epsilon()

        valid_candidates = [
            aid for aid in candidate_article_ids
            if aid in article_embedding_dict
        ]

        if len(valid_candidates) == 0:
            return None

        if np.random.rand() < epsilon:
            return np.random.choice(valid_candidates)

        best_score = -np.inf
        best_article = None

        for aid in valid_candidates:
            article_emb = article_embedding_dict[aid]
            x = np.concatenate([user_features, article_emb])

            score = self.predict(x)

            if score > best_score:
                best_score = score
                best_article = aid

        return best_article

    def update(self, user_features, chosen_article_id, reward, article_embedding_dict, lr=0.01):
        article_emb = article_embedding_dict[chosen_article_id]
        x = np.concatenate([user_features, article_emb])

        pred = self.predict(x)
        error = reward - pred

        self.theta += lr * error * x

        self.t += 1

In [ ]:
class LinUCB:
    def __init__(self, d, alpha=1.0):
        self.name = "LinUCB"
        self.d = d
        self.alpha = alpha
        
        self.A = np.eye(d)
        self.b = np.zeros(d)

    def select_arm(self, user_features, candidate_article_ids, article_embedding_dict):
        valid_candidates = [
            aid for aid in candidate_article_ids
            if aid in article_embedding_dict
        ]
        
        if len(valid_candidates) == 0:
            return None
        
        A_inv = np.linalg.inv(self.A)
        theta = A_inv @ self.b
        
        best_score = -np.inf
        best_article = None
        
        for aid in valid_candidates:
            article_emb = article_embedding_dict[aid]
            x = make_context(user_features, article_emb)
            
            mean_reward = x @ theta
            uncertainty = np.sqrt(x @ A_inv @ x)
            score = mean_reward + self.alpha * uncertainty
            
            if score > best_score:
                best_score = score
                best_article = aid
        
        return best_article

    def update(self, user_features, chosen_article_id, reward, article_embedding_dict):
        article_emb = article_embedding_dict[chosen_article_id]
        x = make_context(user_features, article_emb)
        
        self.A += np.outer(x, x)
        self.b += reward * x

In [ ]:
class LinearThompsonSampling:
    def __init__(self, d, v=0.1):
        self.name = "LinearThompsonSampling"
        self.d = d
        self.v = v
        
        self.A = np.eye(d)
        self.b = np.zeros(d)

    def select_arm(self, user_features, candidate_article_ids, article_embedding_dict):
        valid_candidates = [
            aid for aid in candidate_article_ids
            if aid in article_embedding_dict
        ]
        
        if len(valid_candidates) == 0:
            return None
        
        A_inv = np.linalg.inv(self.A)
        mu = A_inv @ self.b
        cov = (self.v ** 2) * A_inv
        
        theta_sample = np.random.multivariate_normal(mu, cov)
        
        best_score = -np.inf
        best_article = None
        
        for aid in valid_candidates:
            article_emb = article_embedding_dict[aid]
            x = make_context(user_features, article_emb)
            
            score = x @ theta_sample
            
            if score > best_score:
                best_score = score
                best_article = aid
        
        return best_article

    def update(self, user_features, chosen_article_id, reward, article_embedding_dict):
        article_emb = article_embedding_dict[chosen_article_id]
        x = make_context(user_features, article_emb)
        
        self.A += np.outer(x, x)
        self.b += reward * x

In [ ]:
def test_algo(sim, algo_factories, interaction_df, article_embedding_dict, embedding_dim=384):
    results = []

    horizon = len(interaction_df)

    for s in range(sim):
        # fresh algorithms for each simulation
        algos = [factory() for factory in algo_factories]

        for algo in algos:
            cumulative_reward = 0.0

            for t in range(horizon):
                row = interaction_df.iloc[t]

                # 1. user embedding from history
                user_features = get_user_embedding(
                    row["History"],
                    article_embedding_dict,
                    embedding_dim=embedding_dim
                )

                # 2. extract candidate arms directly from current row impression
                impression = row["Impression"]

                if pd.isna(impression) or impression == "":
                    continue

                impression_items = impression.split()

                candidate_article_ids = []
                clicked_article_id = None

                for item in impression_items:
                    # item format: "Nxxxxx-0" or "Nxxxxx-1"
                    article_id, label = item.split("-")
                    candidate_article_ids.append(article_id)

                    if label == "1":
                        clicked_article_id = article_id

                # 3. algorithm chooses arm from current candidate set
                chosen_article_id = algo.select_arm(
                    user_features,
                    candidate_article_ids,
                    article_embedding_dict
                )

                if chosen_article_id is None:
                    continue

                # 4. compute reward
                reward = 1 if chosen_article_id == clicked_article_id else 0

                # 5. update algorithm
                algo.update(
                    user_features,
                    chosen_article_id,
                    reward,
                    article_embedding_dict
                )

                # 6. track cumulative reward
                cumulative_reward += reward

                results.append({
                    "Simulation": s + 1,
                    "Algorithm": algo.name,
                    "Timestep": t + 1,
                    "Chosen_Arm": chosen_article_id,
                    "Reward": reward,
                    "Cumulative_Reward": cumulative_reward
                })

    return pd.DataFrame(results)

In [ ]:
simulation = 20
context_dim = article_embeddings.shape[1] * 2

algo_factories = [
    lambda: DecayingEpsilonGreedyLinear(d=context_dim, initial_epsilon=1.0, decay_rate=0.001),
    lambda: LinUCB(d=context_dim, alpha=1.0),
    lambda: LinearThompsonSampling(d=context_dim, v=0.1)
]

results_df = test_algo(
    sim=simulation,
    algo_factories=algo_factories,
    interaction_df=users_interaction,
    article_embedding_dict=article_embedding_dict,
    embedding_dim=384
)

In [ ]:
result_df["cumulative_reward"] = (
    result_df
    .sort_values(["algo", "Simulation", "t"])
    .groupby(["algo", "Simulation"])["reward_at_t"]
    .cumsum()
)

avg_cum_reward = (
    result_df.groupby(["algo", "t"])["cumulative_reward"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8,5))

for algo in avg_cum_reward["algo"].unique():
    data = avg_cum_reward[avg_cum_reward["algo"] == algo]

    if algo == "BayesUCBQuantile":
        plt.plot(data["t"], data["cumulative_reward"], label=algo, alpha=0.9)
    else:
        plt.plot(data["t"], data["cumulative_reward"], label=algo, alpha=0.5)

plt.xlabel("Time Step")
plt.ylabel("Average Cumulative Reward")
plt.title("Average Cumulative Reward Across Simulations")
plt.legend()
plt.show()